# Class Imbalance: Why 99% Accuracy Can Be Useless

## One-hour machine learning case study

A payment platform builds a fraud detector. Only 1% of transactions are fraudulent. A model predicts “not fraud” for every transaction and achieves 99% accuracy.

Should the company celebrate?

This notebook examines class imbalance, baseline models, precision, recall, F1, precision-recall trade-offs, and resampling inside a valid pipeline.

> **Central question:** Does the metric reflect the problem we actually care about?

## Learning objectives

Students will:

- identify why accuracy can mislead on imbalanced data;
- compare a majority baseline with useful classifiers;
- interpret precision, recall, F1, and confusion matrices;
- connect false positives and false negatives to business consequences;
- use class weighting correctly;
- avoid resampling before the train-test split;
- use AI to challenge metric choices;
- recommend an evaluation strategy, not just one score.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, precision_recall_curve, average_precision_score
)

X, y = make_classification(
    n_samples=10000,
    n_features=15,
    n_informative=7,
    n_redundant=3,
    weights=[0.99, 0.01],
    flip_y=0.002,
    class_sep=1.1,
    random_state=42
)

X = pd.DataFrame(X, columns=[f"feature_{i+1}" for i in range(X.shape[1])])
y = pd.Series(y, name="fraud")

y.value_counts().to_frame("Transactions")

# Part 1 — The misleading success

A model predicts every transaction as legitimate.

Before running the cell:

1. What accuracy do you expect?
2. How many fraud cases will it detect?
3. Which metric will expose the problem?
4. What are the costs of false positives and false negatives?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

def metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

pd.Series(metrics(y_test, dummy_pred)).to_frame("Majority baseline").style.format("{:.2%}")

In [ ]:
pd.DataFrame(
    confusion_matrix(y_test, dummy_pred),
    index=["Actual legitimate", "Actual fraud"],
    columns=["Predicted legitimate", "Predicted fraud"]
)

# Part 2 — Train a standard logistic model

In [ ]:
standard_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

standard_model.fit(X_train, y_train)
standard_pred = standard_model.predict(X_test)

standard_metrics = metrics(y_test, standard_pred)
pd.Series(standard_metrics).to_frame("Standard logistic regression").style.format("{:.2%}")

# Part 3 — Use class weighting

Class weighting tells the algorithm that mistakes on the minority class deserve more attention.

In [ ]:
weighted_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

weighted_model.fit(X_train, y_train)
weighted_pred = weighted_model.predict(X_test)

weighted_metrics = metrics(y_test, weighted_pred)

comparison = pd.DataFrame({
    "Majority baseline": metrics(y_test, dummy_pred),
    "Standard model": standard_metrics,
    "Class-weighted model": weighted_metrics
})

comparison.style.format("{:.2%}")

In [ ]:
pd.DataFrame(
    confusion_matrix(y_test, weighted_pred),
    index=["Actual legitimate", "Actual fraud"],
    columns=["Predicted legitimate", "Predicted fraud"]
)

## Interpretation

1. Which model has the highest accuracy?
2. Which finds the most fraud?
3. Which generates the most false alarms?
4. Which model is “best,” and what information is missing before answering?

# Part 4 — Examine probability scores

Binary predictions hide the ranking produced by the model. Precision-recall curves show the trade-off across many cutoffs.

In [ ]:
weighted_scores = weighted_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, weighted_scores)
ap = average_precision_score(y_test, weighted_scores)

print(f"Average precision: {ap:.3f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve")
plt.grid(True, alpha=0.3)
plt.show()

## Metric selection

For fraud detection, discuss:

- When recall should be prioritized.
- When precision should be prioritized.
- Why the financial amount of fraud may matter more than case count.
- Why one global metric may hide customer or merchant differences.

# Part 5 — Resampling warning

Oversampling or undersampling must happen **inside the training process**, never before the train-test split.

Otherwise, synthetic or duplicated observations can contaminate evaluation data and create unrealistic performance.

Even when resampling is used, the untouched test set should preserve the real class distribution.

## AI as a skeptical reviewer

Use one prompt after choosing your preferred evaluation metrics:

> Challenge my choice of accuracy, precision, recall, and F1 for fraud detection.

> What business information is missing from this confusion matrix?

> Explain why resampling before the train-test split is dangerous.

> What subgroup checks should be performed before deployment?

Evaluate:

**Useful challenge:**  

**Unsupported assumption:**  

**Additional metric needed:**  

**Decision I will revise:**

# Transfer task — Disease screening

A condition affects 0.5% of the population.

A model has:

- 99% accuracy;
- 5% recall;
- 80% precision.

Answer:

1. Why is accuracy not enough?
2. What does 5% recall mean for patients?
3. What would happen if the threshold were lowered?
4. What costs should be considered?
5. Which metric would you monitor most closely, and why?

# Exit reflection

- The majority baseline is important because …
- Accuracy failed because …
- Recall answers …
- Precision answers …
- A useful metric must reflect …

# Instructor checklist

- [ ] Students compared against a majority baseline.
- [ ] Students interpreted confusion-matrix counts.
- [ ] Students connected metrics to real consequences.
- [ ] Students compared class weighting with the standard model.
- [ ] Students understood resampling leakage.
- [ ] AI challenged metric selection.
- [ ] Students transferred the concept to healthcare.

# Closing principle

A metric is useful only when it reflects the errors that matter.

On imbalanced problems, a high accuracy score may simply describe how rare the important cases are.